# Einsatzdaten der Berliner Feuerwehr

## Einleitung und Fragestellung

Wird die Einhaltung der Hilfsfrist bei Einsätzen der Berliner Feuerwehr durch
- die Art des Einsatzes,
- die Systemlast der Leitstelle
- dem Standort (nach Berliner Bezirken)
- oder externe Faktoren wie das Wetter beeinflusst?

Hierfür haben wir uns den Open-Data Datensatz der Berliner Feuerwehr sowie Wetterdaten analysiert.
Die Berliner Feuerwehr stellt seit fast zehn Jahren Daten aus Einsätzen öffentlich zur Verfügung und unterstützt damit die Open-Data Strategie Berlins.
Darunter sind zum Beispiel allgemeine Einsatzdaten, Dispatchcodes, Regionale Daten, Ausrückzeiten und auch Daten aus den Leitstellen und der täglichen Abgabe an die Kassenärztliche Vereinigung Berlins.

Vor allem in Hinblick auf die Einschnitte im Gesundheitssystem<sup>1</sup> (https://www.tagesschau.de/inland/innenpolitik/reform-gesundheitssystem-warken-100.html), die nicht ausreichende Digitalisierung im Gesundheitsbereich<sup>2</sup> (https://www.aok.de/pp/gg/update/schleppende-digitalisierung/) und dem Fachkräftemangel<sup>3</sup> () sind öffentliche Daten ein wichtiges Mittel um demokratische Prozesse zu unterstützen und der Gesellschaft eine Möglichkeit zu geben, Transparenz zu schaffen.
Dabei gibt es für die Hilfsfrist in Deutschland keinen gesetzlichen Standard, heißt die Empfehlung von acht Minuten Ausrückzeit für einen lebensbedrohlichen Einsatz ist in vielen Bereichen Deutschlands nicht einheitlich geregelt. 
So auch in Berlin. Hier gibt es keine gesetzlich bindende Hilfsfrist, sondern ein vereinbartes "Schutzziel". Die Berliner Feuerwehr selbst nennt 90% Erreichungsgrad in 10 bzw. 11 Minuten je nach Notfallart. 
Aufgrund der medizinischen Relevanz der 8-minütigen Hilfsfrist bei lebensbedrohlichen Einsätzen wurde für die Analyse diese Grenze gewählt. Diese Grenze begründet sich alleine aus der Tatsache, dass schon nach wenigen Minuten irreversible Hirnschäden bei Kreislauf-/Atemstillstand auftreten können, da die Sauerstoffversorgung unterbrochen ist. 
Diese Wahl deckt sich außerdem mit der Empfehlung einer Organisationsuntersuchung des Rettungsdienstes im Land Berlin von 2016: statt des damals geltenden, regional gestaffelten Schutzziels (8 Minuten, erreicht in 75% der Fälle im zentralen Berlin bzw. 50% in ländlichen Gebieten) empfiehlt die Studie ein einheitliches Ziel von mindestens 90% aller Notfälle innerhalb von 8 Minuten — und zwar für alle Einsätze, nicht nur die dringendsten<sup>4</sup> (siehe https://www.berliner-feuerwehr.de/fileadmin/bfw/dokumente/status-5/G668_Orga-RD_Berlin__Stand_22.07.16_.pdf).
Somit sind unsere Zahlen zwar strenger als Berlins eigenes, aktuelles Schutzziel (90% in 10 bzw. 11 Minuten), decken sich aber mit einer fachlich begründeten Empfehlung und unterstreichen zusätzlich das Problem in Rettungseinrichtungen.
Auf Empfehlung der Björn Steiger Stiftung sollen Hilfsfristen gestaffelt werden. Leider war bis zum heutigen Datum (18. Juni 2026) die Empfehlung der Björn Steiger Stiftung über die Hilfsfristen je nach Einsatzart noch nicht veröffentlicht<sup>5</sup> (siehe https://rettungslandschaft.steiger-stiftung.de/wann-kommt-steiger-hilfsfristmodell/).

Für die Wetterdaten haben wir die REST-API von Open Meteo genutzt.

### Hypothesen

1. Die Art des Einsatzes (z.B. kritischer Rettungsdienst vs. Brand) beeinflusst die Einhaltung der Hilfsfrist unterschiedlich stark — kritische Einsätze werden priorisiert und sollten häufiger eingehalten werden.
2. Eine hohe Systemlast der Leitstelle (viele unbeantwortete Anrufe, lange Antwortzeiten) korreliert mit einer schlechteren Hilfsfrist-Einhaltung.
3. Extreme Wetterlagen (Stürme, Starkregen, Hitze) erhöhen sowohl die Systemlast der Leitstelle als auch die Wahrscheinlichkeit, dass die Hilfsfrist verfehlt wird — Wetter wirkt also vermutlich indirekt über die Leitstelle.

Einschränkung: Leitstellen-/Anrufdaten sind erst ab 2024 verfügbar. Der gemeinsame Datensatz für die Modellierung ist deshalb auf 2024–2025 beschränkt, auch wenn Einsatz- und Wetterdaten weiter zurückreichen.

# Datenimport

- Laden der täglichen Einsatz- und Anrufdaten der Berliner Feuerwehr aus dem BF-Open-Data-Repository
- Laden der Wetterdaten über die REST-API von Open-Meteo (Vorlage für Code stammt von Open Meteo)
- Überführen aller Daten in geeignete Formate (z.B. Datumsspalten) für die spätere Analyse
- Start- und Enddatum für die Analyse manuell auf 2022–2025 festgelegt, um COVID-19 Ausreißer auszuschließen

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Start- und Enddatum
start_date = pd.Timestamp("2022-01-01")
end_date = pd.Timestamp("2025-12-31")

# daily_mission Daten
df_daily_mission = pd.read_csv("data/BFw_mission_data_daily.csv")
# Geeignetes Datenformat wählen
df_daily_mission["mission_created_date"] = pd.to_datetime (df_daily_mission["mission_created_date"])
df_daily_mission = df_daily_mission.set_index("mission_created_date").sort_index()


# daily_call Daten
df_daily_call = pd.read_csv("data/BFw_call_data_daily.csv")
# Geeignetes Datenformat wählen
df_daily_call["calls_received_day"] = pd.to_datetime(df_daily_call["calls_received_day"])
df_daily_call = df_daily_call.sort_values("calls_received_day").set_index("calls_received_day")
# call_count als str eingelesen → in int umwandeln
df_daily_call["call_count"] = pd.to_numeric(df_daily_call["call_count"], errors="coerce")

# mission_data Daten
df_mission_data = pd.concat([
    pd.read_csv("data/mission_data_set_open_data_2022.csv", low_memory=False),
    pd.read_csv("data/mission_data_set_open_data_2023.csv", low_memory=False),
    pd.read_csv("data/mission_data_set_open_data_2024.csv", low_memory=False),
    pd.read_csv("data/mission_data_set_open_data_2025.csv", low_memory=False),], ignore_index=True)

# Geeignetes Datenformat wählen
df_mission_data["mission_date"] = pd.to_datetime(df_mission_data["mission_date"])
df_mission_data = df_mission_data.set_index("mission_date").sort_index()

# Filter für die Daten
df_daily_mission = df_daily_mission.loc[start_date:end_date]
df_daily_call = df_daily_call.loc[start_date:end_date]
df_mission_data = df_mission_data.loc[start_date:end_date]

In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"start_date": "2022-01-01",
	"end_date": "2025-12-31",
	"daily": ["temperature_2m_max", "weather_code", "precipitation_sum", "snowfall_sum", "wind_speed_10m_max"],
	"timezone": "Europe/Berlin",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
daily_weather_code = daily.Variables(1).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(2).ValuesAsNumpy()
daily_snowfall_sum = daily.Variables(3).ValuesAsNumpy()
daily_wind_speed_10m_max = daily.Variables(4).ValuesAsNumpy()

daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["weather_code"] = daily_weather_code
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["snowfall_sum"] = daily_snowfall_sum
daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max

df_weather = pd.DataFrame(data = daily_data)
df_weather = df_weather.set_index("date")
df_weather.index = df_weather.index.tz_localize(None)
print("\nDaily data\n", df_weather)



# Datenverständnis

Bevor mit der Bereinigung und Analyse begonnen wird, verschaffen wir uns einen Überblick über die geladenen Datensätze: wie viele Einsätze/Tage liegen vor, welche Spalten und Datentypen gibt es, und wo gibt es fehlende Werte (NaN)? Das bildet die Grundlage für die Entscheidungen in der folgenden Datenbereinigung.

In [ ]:
# mission data
print(df_mission_data.shape)
print(df_daily_mission.dtypes)
df_mission_data.head()

#missing values
df_mission_data.isnull().sum()

#zielvariable prüfen
print(df_mission_data["response_time"].describe())
print((df_mission_data["response_time"] <= 480).mean())

Über 2 Millionen Einsätze, 16 Variabeln

8-Minuten-Schutzziel (480 Sekunden) — keine gesetzliche Hilfsfrist, sondern ein politisch vereinbartes Schutzziel mit 75%-Zielquote (siehe Korrektur oben):
Mean: 677 Sekunden (ca 11 Minuten)
Median: 625 Sekunden (ca 10 Minuten)
Min: 121 Sekunden
Max: 1919 Sekunden (ca 32 Minuten)
Count: ca 264.000 Einträge haben keinen Wert (NaN) --> ca 12% muss bereinigt werden
17,2% der Einsätze halten das 8-Minuten-Schutzziel ein


In [ ]:
#daily mission data
print(df_daily_mission.shape)
print(df_daily_mission.dtypes)
df_daily_mission.head()

#missing values
df_daily_mission.isnull().sum()

#zielvariable prüfen
print(df_daily_mission["response_time_ems_critical_mean"].describe())
print(df_daily_mission["response_time_ems_critical_cpr_mean"].describe())

Critical EMS:
- Mean: 642 Sekunden (ca. 10.7 Minuten)
- Min: 558 Sekunden (ca. 9.3 Minuten) !!!

Critical EMS CPR:
- Mean: 505 Sekunden (ca. 8.4 Minuten)
- Min: 360 Sekunden (6 Minuten)

CPR Einsätze werden schneller beantwortet als allgemein kritische Fälle. Annahme: Priorisierungssystem der Leistelle funktioniert. CPR hat höchste prio.

In [ ]:
#daily calls
print(df_daily_call.shape)
print(df_daily_call.dtypes)
df_daily_call.head()

#missing values
df_daily_call.isnull().sum()

#zielvariable prüfen
print(df_daily_call["answer_time_mean_112"].describe())
print(df_daily_call["call_count_112"].describe())
print(df_daily_call["call_count_112_not_answered"].describe())
print(df_daily_call["call_count_112_answered_after_60_seconds"].describe())
print(df_daily_call["call_count_112_answered_within_10_seconds"].describe())

#Für die Systemauslastung
#Anteil nicht beantworteter Anrufe
#call_count_112 zählt laut Datenbeschreibung nur beantwortete Anrufe → korrekter Nenner: call_count_112 + call_count_112_not_answered
print("not answered rate")
df_daily_call["not_answered_rate"] = df_daily_call["call_count_112_not_answered"] / (df_daily_call["call_count_112"] + df_daily_call["call_count_112_not_answered"])
print(df_daily_call["not_answered_rate"].describe())

#Anteil nach 60 Sekunden beantwortet
print("after 60 rate")
df_daily_call["after_60s_rate"] = df_daily_call["call_count_112_answered_after_60_seconds"] / df_daily_call["call_count_112"]
print(df_daily_call["after_60s_rate"].describe())

In [ ]:
from scipy import stats

#Ausreißer mit Z score herausfinden (claude gefragt)
z_score = stats.zscore(df_daily_call["not_answered_rate"])
ausreisser = df_daily_call[abs(z_score) > 3]
print(ausreisser.sort_values("not_answered_rate", ascending = False))

Die Z-Score-Ausreißer (`|z|>3`) bei `not_answered_rate` und `after_60s_rate` lassen sich größtenteils erklären:

- **01.01.2025**: laut README-Disclaimer der Berliner Feuerwehr gab es zum Jahreswechsel 24/25 eine technische Einschränkung der 112-Rufnummer, Notrufe liefen über ein Fallback-System ohne automatisierte Erfassung. Die Zahlen für diesen Tag sind unvollständig → **wird aus der Analyse entfernt**, da es sich um einen Datenfehler statt um eine echte Lastspitze handelt.
- **23.06. und 26.06.2025**: an beiden Tagen gab es laut Open-Meteo-Daten schwere Gewitter mit Windböen von 85–88 km/h (weather_code 95/96) nach einer Hitzewelle. Das erklärt den Anstieg unbeantworteter Notrufe plausibel und **belegbar** → bleibt in den Daten, dient sogar als Beispiel für den vermuteten Wetter→Leitstelle-Zusammenhang.
- **07.09. und 20.11.2024**: keine Wetterauffälligkeiten an diesen Tagen. Mögliche Erklärung: Großveranstaltungen (07.09. Start von Lollapalooza Berlin, 20.11. diverse Konzerte) könnten zu mehr Notrufen geführt haben — diese Erklärung ist aber **nicht belegt**, da in einer Stadt mit 3,8 Mio. Einwohnern an fast jedem Tag irgendwelche Veranstaltungen stattfinden. Die Tage bleiben in den Daten, werden aber als ungeklärt dokumentiert statt kausal erklärt.

FEHLER MEINERSEITS (aus dem vorherigen Durchgang): die `call_count`-Spalten zählen nur die *beantworteten* Anrufe, nicht alle eingegangenen — das hat die ursprüngliche Berechnung der Rate verzerrt, ist hier aber bereits korrigiert (Nenner = `call_count_112 + call_count_112_not_answered`).

Antwortzeit wird von der Björn Steigerstiftung für kritische fälle
answer_time_mean_112:
- Mean: 26 Sekunden
- Max: 68 Sekunden

not_answered_rate:
mean 34,6%
Max 121% (?)

after_60s_rate:
- mean 13,2%
- max 40,4%


In [ ]:
#Wetterdaten
print(df_weather.shape)
print(df_weather.dtypes)
df_weather.head()

#missing values
print(df_weather.isnull().sum())

#zielvariable prüfen
print(df_weather.describe())

# Datenbereinigung

**Zeitraum:** Mission- und Wetterdaten reichen weiter zurück (2018 bzw. 2022), aber `df_daily_call` existiert erst seit 2024. Da die Leitstellen-Auslastung zentral für unsere Fragestellung ist, beschränken wir den gemeinsamen Datensatz für die Modellierung auf 2024–2025. Explorative Auswertungen, die nur Mission- oder Wetterdaten brauchen, können trotzdem den längeren Zeitraum nutzen.

**Wetterdaten-Bug:** Durch die UTC→Europe/Berlin-Konvertierung landet der Zeitindex von `df_weather` auf 23:00 Uhr des Vortags statt auf Mitternacht. Vor dem Join wird der Index korrigiert, sonst würden Wetterdaten dem falschen Tag zugeordnet werden.

**response_time NaN:** ca. 12% der Einsätze (≈265.000 von 1,86 Mio.) haben keinen Wert bei `response_time`. Diese Zeilen werden entfernt, da sich für sie keine Zielvariable berechnen lässt.

**response_time Ausreißer werden NICHT entfernt:** Ein Z-Score-Test zeigt ca. 36.000 Einsätze (~2%) mit ungewöhnlich langer Reaktionszeit (1455–1919 Sekunden). Bei genauer Betrachtung sammeln sich die Außreißer vor allem bei nicht lebensbedrohlichen Einsätzen an. Sie zu entfernen würde die Zielvariable künstlich verzerren.

**dispatchcode_criticality / dispatchcode_category NaN:** ca. 111.000 Einsätze haben keinen Dispatchcode. Statt diese Zeilen zu droppen was einen hohen Datenverlust bedeuten würde, wird eine eigene Kategorie "unbekannt" eingeführt.

**Zielvariable:** `hilfsfrist = response_time <= 480` (480 Sekunden = Hilfsfrist, **keine gesetzliche Hilfsfrist von Berlin**)

**Zeitliche Features:** aus `mission_date` werden Wochentag und Monat extrahiert. Eine Uhrzeit-Variable ("Stunde") ist nicht möglich, da `mission_date` in allen Jahresdateien (2018–2026) ausschließlich das Datum ohne Uhrzeitkomponente enthält — keiner der verfügbaren BF-Open-Data-Datensätze (Mission_Data, Daily_Data, Turnout_Times, KV_Data, Regional_Data) liefert eine Uhrzeit-Auflösung. Die Granularität der Daten kann die Analyse einschränken.


In [ ]:
# Wetter anpassen, Zeitstempel-Bug korrigieren
df_weather.index = (df_weather.index + pd.Timedelta(hours=1)).normalize()

# Kontrolle
print(df_weather.index.min(), df_weather.index.max())
print(df_weather.head())
print(df_weather.tail())

In [ ]:
# response_time NaN entfernen
print("vorher:", df_mission_data.shape)
# dropna löscht Zeilen wo NaN ist
df_mission_data = df_mission_data.dropna(subset=["response_time"])
print("nachher", df_mission_data.shape)

In [ ]:
# Zielvariable Hilfsfrist eingehalten
df_mission_data["hilfsfrist"] = df_mission_data["response_time"] <= 480
print(df_mission_data["hilfsfrist"].value_counts(normalize=True))

# Wochentag und Monat aus dem Datums-Index rausziehen
df_mission_data["wochentag"] = df_mission_data.index.day_name()
df_mission_data["monat"] = df_mission_data.index.month

print(df_mission_data[["wochentag", "monat"]].head())

### Kategoriale Variablen encodieren

Modelle rechnen nur mit Zahlen, nicht mit Text wie `"Brand"` oder `"Mitte"`.

- **NaN bei `dispatchcode_criticality`** → wird zu eigener Kategorie `"unbekannt"` (statt Zeilen zu löschen).
- **One-Hot-Encoding** für `mission_type`, `mission_location_district`, `dispatchcode_criticality`: jede Kategorie wird eine eigene 0/1-Spalte. So entsteht keine künstliche Rangordnung zwischen den Kategorien (z.B. Bezirken), wie es bei einer einfachen Durchnummerierung der Fall wäre.
- Ergebnis: mehr Spalten, gleich viele Zeilen.

In [ ]:
# NaN bei Dispatchcode als eigene Kategorie "unbekannt"
df_mission_data["dispatchcode_criticality"] = df_mission_data["dispatchcode_criticality"].fillna("unbekannt")

# One-Hot-Encoding (für Modelltraining wichtig)
df_mission_data = pd.get_dummies(
    df_mission_data,
    columns=["mission_type", "mission_location_district", "dispatchcode_criticality"],
    prefix=["typ", "district", "criticality"],
)

print(df_mission_data.shape)
df_mission_data.head()

# Nach Rettungsdienst-Einsätze + Rettungseinsätze mit technischer Hilfsleistung filtern für Hilfsfrist 
print("vorher: ", df_mission_data.shape)
df_mission_data = df_mission_data[
    (df_mission_data["typ_Rettungsdienst"] == True) |
    (df_mission_data["typ_Rettungsdienst mit Technischer Hilfeleistung"] == True)
]
print("nachher:", df_mission_data.shape)

In [ ]:
# Silvester-Tag entfernen: dokumentierter Systemausfall (README-Disclaimer)
print("vorher:", df_daily_call.shape)
df_daily_call = df_daily_call.drop(pd.Timestamp("2025-01-01"))
print("nachher:", df_daily_call.shape)

# Explorative Analyse: Vorgehen

Wir schauen uns die Daten in vier Stufen an, statt direkt mit komplexen Visualisierungen zu starten:

1. **Langer Zeitraum (2022–2025), einzeln:** Einsatztyp, Wetter, Wochentag, Monat — je für sich gegen `hilfsfrist`. Soll zeigen welche Variabeln einen Effekt haben.
2. **Langer Zeitraum, kombiniert:** Nur für Variablen, die in Stufe 1 einzeln schon ein Muster zeigen.
3. **Kurzer Zeitraum (2024–2025), einzeln:** Leitstellen-Variablen (`not_answered_rate`, `answer_time_mean_112`, ...) gegen `hilfsfrist`/`response_time`. Geht nicht über den langen Zeitraum, da Leitstellendaten erst ab 2024 existieren.
4. **Kurzer Zeitraum, kombiniert:** Leitstelle zusammen mit Wetter bzw. Einsatztyp

## Hilfsfrist VS Dringlichkeitsstufe

**Befund:** Es ist sichtbar, dass es eine Korrelation zwischen der Einhaltung der Hilfsfrist und den Dringlichkeitsstufen (AMPDS) vorhanden ist.

In [ ]:
# Hilfsfrist-Quote nach Dringlichkeitsstufe
crit_spalten = [col for col in df_mission_data.columns if col.startswith("criticality_")]
quote_je_crit = {
    col.replace("criticality_", ""): df_mission_data.loc[df_mission_data[col], "hilfsfrist"].mean()
    for col in crit_spalten
}
quote_je_crit = pd.Series(quote_je_crit)

# Nach AMPDS-Dringlichkeit sortieren (O < A < B < C < D < E), statt nach Quote
reihenfolge = ["O", "A", "B", "C", "D", "E", "unbekannt"]
quote_je_crit = quote_je_crit.reindex([k for k in reihenfolge if k in quote_je_crit.index])

print(quote_je_crit)

quote_je_crit.plot(kind="bar")
plt.ylabel("Hilfsfrist-Quote")
plt.xlabel("Dringlichkeitsstufen")
plt.title("Einhaltung der Hilfsfrist im Rettungsdienst")
plt.xticks(rotation=0)

# Diagramm Beschriftung
plt.figtext(
    0.5, -0.05,
    "AMPDS-Dringlichkeitsstufen (steigend): O=Omega (niedrig) < A=Alpha < B=Bravo < C=Charlie < D=Delta < E=Echo (lebensbedrohlich)",
    wrap=True, ha="center", fontsize=8
)

plt.tight_layout()
plt.show()

## Wetter vs. Hilfsfrist (Stufe 1, einzeln)

Einsatzdaten (nur Rettungsdienst, langer Zeitraum 2022–2025) wurden auf das Tagesdatum mit den Wetterdaten gejoint. Es wurde nach `hilfsfrist` (eingehalten/nicht eingehalten) verglichen.

**Befund:** Die Verteilungen von Temperatur, Niederschlag und Windgeschwindigkeit sind für beide Gruppen (Hilfsfrist eingehalten vs. nicht eingehalten) nahezu identisch.

**Einordnung:** Das bedeutet nicht, dass Wetter keine Rolle spielt, keinen klaren direkten Zusammenhang mit der Hilfsfrist zeigt.

In [ ]:
import seaborn as sns
# Einsatzdaten mit Wetter joinen
df_weather_mission = df_mission_data.join(df_weather, how="left")

print(df_weather_mission.shape)
print(df_weather_mission[["temperature_2m_max", "precipitation_sum", "wind_speed_10m_max"]].isnull().sum())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

variabeln = ["temperature_2m_max", "precipitation_sum", "wind_speed_10m_max"]
titel = ["Temperatur (°C)", "Niederschlag (mm)", "Windgeschwindigkeit (km/h)"]

for ax, var, t in zip(axes, variabeln, titel):
    sns.boxplot(data=df_weather_mission, x="hilfsfrist", y=var, ax=ax)
    ax.set_title(t)
    ax.set_xlabel("Hilsfrist eingehalten")

plt.tight_layout()
plt.show()

## Wochentag & Monat VS. Hilfsfrist

**Befund Monat (einzeln):** Die Hilfsfrist-Quote schwankt nur minimal über die Monate heißt es gibt kein erkennbares saisonales Muster. 

**Befund Wochentag (einzeln):** Am Wochenende wird die Hilfsfrist etwas häufiger eingehalten als werktags.

**Befund Heatmap Wochentag × Monat (kombiniert):** Nur geringe Abweichungen zwischen den Zellen, kein zusätzlicher Effekt.

**Einordnung**: Auch gibt es keine klar erkennbare Korrelation zwischen Sasion bzw. Wochentag und Hilfsfrist. Bis auf eine bessere Einhaltung an den Wochenenden, was vielleicht auch auf eine vorrausschauende PLanung zu schließen ist, hat sich nichts markant herausgestellt.

In [ ]:
# Hilfsfrist-Quote pro Monat
quote_monat = df_mission_data.groupby("monat")["hilfsfrist"].mean()

quote_monat.plot(kind="bar")
plt.ylabel("Hilfsfrist eingehalten")
plt.title("Hilfsfrist eingehalten nach Monat")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Hilfsfrist-Quote je Wochentag
quote_wochentag = df_mission_data.groupby("wochentag")["hilfsfrist"].mean()
sort_wochentag = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
quote_wochentag = quote_wochentag.reindex(sort_wochentag)

quote_wochentag.plot(kind="bar")
plt.ylabel("Hilfsfrist eingehalten")
plt.title("Hilfsfrist eingehalten nach Wochentag")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Heatmap Hilfsfrist nach Wochentag, Monat
pivot = df_mission_data.pivot_table(index="wochentag", columns="monat", values="hilfsfrist", aggfunc="mean")

sort_wochentag = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
pivot = pivot.reindex(sort_wochentag)

plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("Hilfsfrist eingehalten nach Wochentag und Monat")
plt.xlabel("Monat")
plt.ylabel("Wochentag")
plt.tight_layout()
plt.show()

## Hilfsfrist VS Bezirk

**Befund:** Die Hilfsfristquote variiert stark zwischen den Bezirken und Kritikalitätsstufen. Während in einzelnen Konstellationen über 50 % der Einsätze die Hilfsfrist einhalten, werden in anderen Fällen nur 0,04 % erreicht. Dies weist auf einen starken Zusammenhang zwischen Bezirk und Hilfsfrist-Einhaltung hin.

In [ ]:
# Hilfsfrist-Quote nach Berliner Bezirk
district_spalten = [col for col in df_mission_data.columns if col.startswith("district_")]
quote_je_bezirk = {
    col.replace("district_", ""): df_mission_data.loc[df_mission_data[col], "hilfsfrist"].mean()
    for col in district_spalten
}
quote_je_bezirk = pd.Series(quote_je_bezirk).sort_values(ascending=False)

print(quote_je_bezirk)
print()
print("Spannweite (max-min):", quote_je_bezirk.max() - quote_je_bezirk.min())

plt.figure(figsize=(10, 5))
quote_je_bezirk.plot(kind="bar")
plt.ylabel("Anteil Hilfsfrist eingehalten")
plt.title("Hilfsfrist-Einhaltung nach Bezirk (Rettungsdienst)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
# Heatmap: Hilfsfrist-Quote nach Bezirk x Dringlichkeitsstufe
crit_spalten = [col for col in df_mission_data.columns if col.startswith("criticality_")]
district_spalten = [col for col in df_mission_data.columns if col.startswith("district_")]

pivot_daten = []
for d_col in district_spalten:
    bezirk = d_col.replace("district_", "")
    zeile = {}
    for c_col in crit_spalten:
        crit = c_col.replace("criticality_", "")
        maske = df_mission_data[d_col] & df_mission_data[c_col]
        zeile[crit] = df_mission_data.loc[maske, "hilfsfrist"].mean()
    pivot_daten.append(pd.Series(zeile, name=bezirk))

pivot_bezirk_crit = pd.DataFrame(pivot_daten)

# Spalten nach AMPDS-Dringlichkeit sortieren
reihenfolge = ["O", "A", "B", "C", "D", "E", "unbekannt"]
pivot_bezirk_crit = pivot_bezirk_crit[[c for c in reihenfolge if c in pivot_bezirk_crit.columns]]
# Nach Gesamtquote sortieren
pivot_bezirk_crit = pivot_bezirk_crit.loc[pivot_bezirk_crit.mean(axis=1).sort_values(ascending=False).index]

plt.figure(figsize=(10, 7))
sns.heatmap(pivot_bezirk_crit, annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("Hilfsfrist-Quote nach Bezirk × Dringlichkeitsstufe")
plt.xlabel("AMPDS-Kritikalität")
plt.ylabel("Bezirk")
plt.tight_layout()
plt.show()

## Leistelle VS Hilfsfrist
**Befund:** Es ist kein klarer Zusammenhang zwischen der Beantwortung der Anrufe durch die Leitstelle und Hilfsfrist auszumachen

In [ ]:
# Einsatzdaten auf Erfassung der Anrufe begrenzen (Jahre) 2024-2025
df_mission_data_cut = df_mission_data.loc[pd.Timestamp("2024-01-01"):pd.Timestamp("2025-12-31")]

# Einsatzdaten joinen mit Anrufdaten
df_leitstelle = df_mission_data_cut.join(df_daily_call, how="left")

print(df_leitstelle.shape)
print(df_leitstelle[["not_answered_rate", "after_60s_rate", "answer_time_mean_112"]].isnull().sum())

# Boxplot: Leistelle VS Hilfsfrist
df_leitstelle_plot = df_leitstelle.reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

leitstellen_var = ["not_answered_rate", "after_60s_rate", "answer_time_mean_112"]
titel = ["Anteil nicht beantwortet", "Anteil nach 60s beantwortet", "Mittlere Antwortzeit (s)"]

for ax, var, t in zip(axes, leitstellen_var, titel):
    sns.boxplot(data=df_leitstelle_plot, x="hilfsfrist", y=var, ax=ax)
    ax.set_title(t)
    ax.set_xlabel("Hilfsfrist eingehalten")

plt.tight_layout()
plt.show()



In [ ]:
# Auf Tagesebene aggregieren statt pro Einsatz (vermeidet Verdünnung durch Einsatzebene-Varianz)
df_tage = df_leitstelle.groupby(df_leitstelle.index)["hilfsfrist"].mean().to_frame("hilfsfrist_quote")
df_tage = df_tage.join(df_daily_call[["not_answered_rate", "after_60s_rate", "answer_time_mean_112"]])

print(df_tage.corr()["hilfsfrist_quote"])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, var in zip(axes, ["not_answered_rate", "after_60s_rate", "answer_time_mean_112"]):
    ax.scatter(df_tage[var], df_tage["hilfsfrist_quote"], alpha=0.5)
    ax.set_xlabel(var)
    ax.set_ylabel("Hilfsfrist-Quote (Tag)")
plt.tight_layout()
plt.show()

In [ ]:
# Einsatzaufkommen pro Tag (Anzahl Rettungsdienst-Einsätze) als direkterer Systemlast-Indikator
df_tage["einsatz_anzahl"] = df_leitstelle.groupby(df_leitstelle.index).size()

print(df_tage.corr()["hilfsfrist_quote"])

# Zusätzlich: hängt not_answered_rate selbst mit dem Einsatzaufkommen zusammen?
print()
print("Korrelation not_answered_rate <-> Einsatzanzahl:", df_tage["not_answered_rate"].corr(df_tage["einsatz_anzahl"]))

## Leitstelle wird nicht weiterverfolgt

**Befund:** `not_answered_rate` korreliert statistisch signifikant mit der täglichen Hilfsfrist-Quote (r=-0,29, p≈0,000000, n=730 Tage). Der Zusammenhang ist aber moderat und somit deutlich schwächer als die Dringlichkeitsstufe oder der Bezirk.

**Einschränkung:** Die Feldbeschreibung definiert `call_count_112_not_answered` nur als "Number of emergency calls (112) not answered", ohne zu erklären, ob zum Beispiel Mehrfachanrufe zum selben Vorfall mitgezählt werden. Damit ist unklar, ob wirklich eine Systemüberlastung der Leitstelle gemessen oder das Anrufverhalten widerspiegelt wird. Die erklärbaren Ausreißertage (Silvester-Systemausfall, Sturmtage) sprechen für eine Aussagekraft, die unerklärten Außreißertage bleiben aber unbelegt.

**Entscheidung:** Aufgrund dieser Unsicherheit verfolgen wir die Leitstellen-Variablen für das Modelltraining nicht weiter. Wir können nicht sauber genug einordnen, was genau gemessen wird.

## Wie weit wird die Hilfsfrist bei Echo-Einsätzen - also Einsätzen mit der höchsten Dringlichkeitsstufe - über/unterschritten?

Es ist anzunehmen, dass die Leistelle die Einsätze genau genug priorisiert, sodass bei Einsätzen bei denen schnell gehandelt werden muss, die Hilfsfrist eingehalten wird.

**Befund:** Bei über 50% der Echo-Einsätze (lebensbedrohliche Situation) wird die Hilfsfrist verfehlt. Auch wenn eine geringe Überschreitung von zwei Minuten vorliegt, iat die Verzögerung medizinisch relevant.

In [ ]:
echo_response = df_mission_data.loc[df_mission_data["criticality_E"], "response_time"]

print(echo_response.describe())
print()
print("Median Überschreitung (nur verfehlte Fälle):", (echo_response[echo_response > 480] - 480).median(), "Sekunden")
print("Mittlere Überschreitung (nur verfehlte Fälle):", (echo_response[echo_response > 480] - 480).mean(), "Sekunden")

plt.figure(figsize=(8, 5))
plt.hist(echo_response, bins=50)
plt.axvline(480, color="red", linestyle="--", label="Hilfsfrist (480s)")
plt.xlabel("Response Time (Sekunden)")
plt.ylabel("Anzahl Einsätze")
plt.title("Verteilung der Reaktionszeit bei Echo-Einsätzen")
plt.legend()
plt.tight_layout()
plt.show()

# Zielvariable & Features finalisieren

Genutzt werden nur Variablen mit belegtem Effekt: Dringlichkeit (AMPDS) und Bezirk, plus Einsatztyp (Rettungsdienst vs. Mischtyp). Wochentag, Monat, Wetter und Leitstelle werden bewusst ausgeschlossen (siehe Begründungen oben).

In [ ]:
# Matrix zusammenstellen (nur Dringlichkeit, Bezirk, Einsatztyp)
feature_spalten = (
    [c for c in df_mission_data.columns if c.startswith("criticality_")] +
    [c for c in df_mission_data.columns if c.startswith("district_")] +
    ["typ_Rettungsdienst", "typ_Rettungsdienst mit Technischer Hilfeleistung"]
)

X = df_mission_data[feature_spalten]
y = df_mission_data["hilfsfrist"]

print(X.shape, y.shape)
print(X.columns.tolist())

# Modelltraining

## Modell 1: Logistische Regression

Nachdem die Matrix steht, trainieren wir zunächst ein einfaches Modell: die Logistische Regression. Sie sagt vorher, ob die Hilfsfrist eingehalten wird (Ja/Nein), basierend auf Dringlichkeit, Bezirk und Einsatztyp. Sie dient als **Ausgangspunkt (Baseline)** — bevor wir komplexere Modelle ausprobieren. Erstmal wollen wir wissen wie ein einfaches Modell abschneidet.

### Variante A: ohne `class_weight` (Baseline)

**Befund:** Es gibt ein Problem in unseren Daten: nur etwa 19,7% der Einsätze halten die Hilfsfrist ein, der Rest (80,3%) nicht. Die Trainingsklassen sind also unausgeglichen. Ein Modell könnte deshalb einfach immer "nicht eingehalten" vorhersagen und hätte damit schon ca. 80% Trefferquote (Accuracy), ohne wirklich etwas gelernt zu haben.

In [ ]:
# Train/Test Split (80/20)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)
print("Anteil hilfsfrist=True (train):", y_train.mean())
print("Anteil hilfsfrist=True (test):", y_test.mean())

# Logistische Regression trainieren
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

print("Trainings-Accuracy:", log_reg.score(X_train, y_train))
print("Test-Accuracy:", log_reg.score(X_test, y_test))

y_pred = log_reg.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

### Variante B: mit `class_weight="balanced"`

Um das Ungleichgewicht auszugleichen, gewichtet diese Variante die seltenere Klasse ("Hilfsfrist eingehalten") stärker. Damit soll das Modell nicht mehr einfach die Mehrheitsklasse raten.

**Befund:** Unseren Erwartungen entsprechend ist das Ergebnis ausgeglichener, aber dennoch nicht ausreichend.

In [ ]:
# Logistische Regression mit class_weight="balanced"
log_reg_balanced = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg_balanced.fit(X_train, y_train)

print("Trainings-Accuracy:", log_reg_balanced.score(X_train, y_train))
print("Test-Accuracy:", log_reg_balanced.score(X_test, y_test))

y_pred_balanced = log_reg_balanced.predict(X_test)
print(confusion_matrix(y_test, y_pred_balanced))
print(classification_report(y_test, y_pred_balanced))

## Modell 2: Random Forest

Als zweites Modell probieren wir einen Random Forest aus. Er kann im Gegensatz zur Logistischen Regression auch nichtlineare Zusammenhänge und Wechselwirkungen abbilden. 

**Befund:** Das Modell ist zwar gegenüber der logistischen Regression besser, aber auch nicht aussagekräftig.

In [ ]:
# Random Forest trainieren
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

print("Trainings-Accuracy:", rf.score(X_train, y_train))
print("Test-Accuracy:", rf.score(X_test, y_test))

y_pred_rf = rf.predict(X_test)
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

## Modell 3: Random Forest Regression
Die vorherigen KLassifikationmodelle haben nur Ja/Nein voraus gesagt. In diesem Ansatz prüfen wir eine voraussage anhand von Zeiteinschätzungen in Sekunden.

**Befund:** Der Random Forest schätzt die Eintreffzeit im Schnitt um ±174 Sekunden. Bei einem Gesamtmedian von ca. 625 Sekunden (10 Minuten) entspricht das einer relativen Abweichung von rund 28%. Für eine exakte Prognose reicht das nicht.

In [ ]:
# Regressionsmodell: response_time (Sekunden) statt hilfsfrist (Ja/Nein) vorhersagen
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

y_minuten = df_mission_data["response_time"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_minuten, test_size=0.2, random_state=42
)

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train_r, y_train_r)

y_pred_r = rf_reg.predict(X_test_r)
print("Mittlerer absoluter Fehler (Sekunden):", mean_absolute_error(y_test_r, y_pred_r))

# Modellbewertung

## Vergleich Log. Regression vs. Random Forest

Jetzt stellen wir alle drei bisher trainierten Klassifikationsmodelle (LogReg unbalanced, LogReg balanced, Random Forest) anhand der gleichen Kennzahlen nebeneinander, um zu sehen, welches am besten mit dem Klassenungleichgewicht umgeht und die Hilfsfrist-Einhaltung am zuverlässigsten vorhersagt.

In [ ]:
# Accuracy, F1, Precision, Recall im direkten Vergleich
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

modelle = {
    "LogReg (unbalanced)": y_pred,
    "LogReg (balanced)": y_pred_balanced,
    "Random Forest": y_pred_rf,
}

vergleich = []
for name, pred in modelle.items():
    vergleich.append({
        "Modell": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision (True)": precision_score(y_test, pred),
        "Recall (True)": recall_score(y_test, pred),
        "F1 (True)": f1_score(y_test, pred),
    })

df_vergleich = pd.DataFrame(vergleich).set_index("Modell")
print(df_vergleich)

## Random Forest: Feature Importances

Welche der drei Eingangsgrößen (Kritikalität, Bezirk, Einsatztyp) nutzt der Random Forest am meisten für seine Vorhersage? Das ist ein direkter Test unserer ersten Hypothese (Einsatzart/Kritikalität beeinflusst die Hilfsfrist) und liefert zusätzlich Hinweise darauf, wie stark der Bezirk im Vergleich dazu ins Gewicht fällt.

In [ ]:
# Welche Features nutzt der Random Forest (Klassifikation) am meisten?
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances)

plt.figure(figsize=(10, 6))
importances.plot(kind="bar")
plt.ylabel("Wichtigkeit")
plt.title("Feature Importances — Random Forest (Hilfsfrist-Klassifikation)")
plt.tight_layout()
plt.show()

# Interpretation

**Welche Prädiktorgruppe erklärt am meisten?**

Modellvergleich (Test-Set):

| Modell | Accuracy | Precision (True) | Recall (True) | F1 (True) |
|---|---|---|---|---|
| LogReg (unbalanced) | 0,80 | 0,58 | **0,01** | 0,02 |
| LogReg (balanced) | 0,58 | 0,26 | 0,61 | **0,37** |
| Random Forest | 0,62 | 0,27 | 0,55 | 0,36 |

Random Forest und balancierte LogReg liegen mit F1≈0,36–0,37 etwa gleichauf — die unbalancierte LogReg ist mit Recall=0,01 wertlos für die eigentliche Fragestellung (erkennt fast keine eingehaltenen Hilfsfristen).

**Feature Importances (Random Forest):** die größten Gewichte liegen auf `criticality_D` (16,5%), `district_Friedrichshain-Kreuzberg` (15,5%), `district_Mitte` (9,7%) und `criticality_E` (7,2%) — Kritikalität und Bezirk dominieren beide, in etwa vergleichbarer Größenordnung. Das bestätigt quantitativ, was die Analyse schon zeigte:

1. **Dringlichkeit (AMPDS)** — stärkster Einzeleffekt (Quote steigt von 9,7% bei O auf 44,8% bei E).
2. **Bezirk** — fast ebenso wichtig im Modell
3. **Bezirk × Dringlichkeit (Heatmap):** die beiden Effekte vermischen sich nicht, sondern addieren sich nur — in jedem einzelnen Bezirk steigt die Quote streng von O zu E (die Kritikalität bestimmt also immer die Reihenfolge), aber das gesamte Niveau verschiebt sich je Bezirk nach oben oder unten (Friedrichshain-Kreuzberg liegt bei jeder Stufe vor Treptow-Köpenick). Wie groß der Bezirksunterschied ist, hängt aber von der Dringlichkeit ab: bei O (niedrigste Stufe) liegen bester und schlechtester Bezirk nur ca. 11 Punkte auseinander (15% vs. 4%), bei E (höchste Stufe) sind es dagegen ca. 40 Punkte (68% vs. 28%) — der Bezirk macht sich also gerade bei den dringendsten Einsätzen am stärksten bemerkbar.

**Wo versagt das Modell?**

- Mit nur drei groben Kategorien bleibt viel unerklärt (nicht in den Daten enthalten: Verkehr, Tageszeit, Fahrzeugverfügbarkeit)
- Klassenungleichgewicht (80/20) lässt die unbalancierte LogReg in die Falle laufen: hohe Accuracy täuscht, sie sagt fast immer "False" vorher
- Mit Gewichtung wird die Vorhersage ehrlicher, aber bleibt weit von einem verlässlichen Einzelfall-Modell entfernt
- Das Regressionsmodell (response_time direkt) hat einen mittleren Fehler von ca. 174 Sekunden. Als grobe Schätzung brauchbar, aber nicht für präzise Einzelfall-Aussagen

# Reflexion & Ausblick

Im folgenden Ergänzen wir Stichpunktartig die **Grenzen der Analyse**:

- 8-Minuten-Schwelle ist medizinisch begründet, aber nicht Berlins offizielles Schutzziel (10/11 Min., 90%), dadurch sind die Zahlen nicht 1:1 mit offiziellen Statistiken vergleichbar
- die Leitstellen-Variablen sind methodisch unklar, weshalb der Effekt im Modell nicht erklärbar ist
- Nur 12 Bezirke als Auflösung: eine feinere Einteilung würde die Analyse verbessern
- Kreislaufstillstand-Kategorie enthält vermutlich auch bereits verstorbene Personen. Die Quote von 55% verpasster Echo-Quote könnte so erklärt werden
- Datenzeiträume uneinheitlich: Mission/Wetter 2022–2025, Leitstelle nur 2024–2025
- Nur Korrelationen geprüft, keine Kausalität 
- Einfache Modelle 
- Keine Uhrzeit-Variable möglich, da die Quelle nur Tagesdaten liefert. Würde die Analyse nochmal verbessern.

**Ausblick und offene Folgefragen:**

- Die Berliner Feuerwehr kann Einsätze, die laut Meldebild nicht zwingend den Rettungsdienst brauchen, an die Kassenärztliche Vereinigung (KV) Berlin abgeben — ein zusätzlicher, in dieser Analyse nicht ausgewerteter Datensatz (`daily_kv_data.csv`) existiert dafür
- Ob und wie stark diese Abgabe die Leitstellen-Auslastung und damit die Hilfsfrist tatsächlich entlastet, ist mit den hier verwendeten Daten/Methoden nicht abschließend zu beantworten — dafür bräuchte es eine eigene Untersuchung
- Offene Folgefragen wären:

    - Wie wirkt sich die KV-Kooperation auf die Einsatzzahlen und die Hilfsfristeinhaltung aus?
    - Wie beeinflussen Bagatellfahrten die Auslastung des Rettungsdienstes?
    - Wie wirkt sich die Leitstellen-Auslastung auf die Hilfsfrist aus?
    - Wie wirken sich die Überlastung der Notaufnahmen, Krankenhäuser und Praxen auf die   Einsatzzahlen aus?

# KI Nutzung im Notebook

Im Rahmen des Projekts wurde Künstliche Intelligenz gezielt als unterstützendes Werkzeug eingesetzt, jedoch nicht als primäre Grundlage für die inhaltliche oder technische Arbeit. Die Bearbeitung der Inhalte erfolgte überwiegend eigenständig auf Basis der Vorlesungsunterlagen sowie ergänzender Literatur- und Internetrecherchen.

Im Verlauf der Entwicklung kamen insbesondere Claude sowie der Claude Agent zum Einsatz. Diese wurden vor allem dann genutzt, wenn bei der Programmierung oder beim Debugging konkrete Probleme auftraten. Dabei ging es insbesondere um das Verständnis von Fehlermeldungen, die Identifikation von Fehlerursachen sowie um Hinweise zur Korrektur einzelner Codebestandteile.

Darüber hinaus wurde Claude bei der Überarbeitung von Texten eingesetzt, insbesondere um Formulierungen zu verbessern und Inhalte klarer und strukturierter darzustellen. Die inhaltliche Ausarbeitung selbst wurde dabei nicht von der KI übernommen, sondern blieb vollständig in eigener Verantwortung.

Der Claude Agent unterstützte zusätzlich bei der Umsetzung der Webanwendung, insbesondere im Bereich der Datenvisualisierung und der technischen Umsetzung einzelner Komponenten. Gerade in diesem Teil des Projekts traten mehrere Herausforderungen auf, die jedoch überwiegend durch eigene Recherche und die Nutzung vorhandener Dokumentationen gelöst werden konnten.

# Quellen

Steiger Stiftung (o. J.): Notrufbearbeitung und Hilfsfrist.
Verfügbar unter: https://rettungslandschaft.steiger-stiftung.de/notrufbearbeitung-und-hilfsfrist/
(Zugriff: 19.06.2026)

Steiger Stiftung (o. J.): Hilfsfristzeiten und Definitionen.
Verfügbar unter: https://rettungslandschaft.steiger-stiftung.de/hilfsfristzeiten-definitionen/
(Zugriff: 19.06.2026)

elab2go (o. J.): Demo Py – Python Umgebung.
Verfügbar unter: https://www.elab2go.de/demo-py/
(Zugriff: 19.06.2026)

Open-Meteo (o. J.): Historical Forecast API Documentation.
Verfügbar unter:
https://open-meteo.com/en/docs/historical-forecast-api
(Zugriff: 19.06.2026)

Berliner Feuerwehr (o. J.): BF Open Data Repository.
Verfügbar unter: https://github.com/Berliner-Feuerwehr/BF-Open-Data
(Zugriff: 19.06.2026)

Berliner Feuerwehr (o. J.): Open Data Service.
Verfügbar unter: https://www.berliner-feuerwehr.de/service/open-data/
(Zugriff: 19.06.2026)

Berliner Feuerwehr (2016): Organisationsstruktur Rettungsdienst Berlin (Stand 22.07.2016).
Verfügbar unter:
https://www.berliner-feuerwehr.de/fileadmin/bfw/dokumente/status-5/G668_Orga-RD_Berlin__Stand_22.07.16_.pdf
(Zugriff: 19.06.2026)

# Vorbereitung für die Visualisierung
## Ab hier haben wir das Notebook mit Vibecoding per Coding Agent (Claude) gestaltet. Dabei haben wir uns **nicht** auf die Ausgaben verlassen sondern uns explizit Vorschläge geben lassen wir unsere Visualisierungsvorstellung umgesetzt werden kann. Die Codezeilen wurden geprüft und kritisch bewertet.

### Die Erkenntnisse wurden verarbeitet, sodass eine Laienfreundliche Visualisierung möglich ist.

### Robustheitscheck: `dispatchcode_category` statt `criticality`

Zusätzlich prüfen wir, ob die feinere `dispatchcode_category` (z.B. "Kreislaufstillstand", "Sturz") die Modellgüte gegenüber der gröberen `criticality`-Stufe wesentlich verändert. Gleicher Bezirk/Einsatztyp, gleicher Split (gleicher `random_state`, gleiches `y_minuten`) — nur das Dringlichkeits-Feature wird ausgetauscht.

In [ ]:
# dispatchcode_category encodieren (bisher rohe String-Spalte, noch nicht one-hot)
df_mission_data["dispatchcode_category"] = df_mission_data["dispatchcode_category"].fillna("unbekannt")
df_mission_data = pd.get_dummies(df_mission_data, columns=["dispatchcode_category"], prefix="kategorie")

# Feature-Matrix mit Kategorie statt Kritikalität (Bezirk/Einsatztyp identisch zu X)
feature_spalten_kategorie = (
        [c for c in df_mission_data.columns if c.startswith("kategorie_")] +
        [c for c in df_mission_data.columns if c.startswith("district_")] +
        ["typ_Rettungsdienst", "typ_Rettungsdienst mit Technischer Hilfeleistung"]
)
X_kategorie = df_mission_data[feature_spalten_kategorie]

# Gleicher Split (gleicher random_state, gleiches y_minuten) wie beim Kritikalitäts-Modell oben
X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(
    X_kategorie, y_minuten, test_size=0.2, random_state=42
)

rf_reg_kategorie = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg_kategorie.fit(X_train_k, y_train_k)

y_pred_k = rf_reg_kategorie.predict(X_test_k)
mae_kategorie = mean_absolute_error(y_test_k, y_pred_k)

print("MAE mit criticality (Minuten):", mean_absolute_error(y_test_r, y_pred_r) / 60)
print("MAE mit dispatchcode_category (Minuten):", mae_kategorie / 60)

In [ ]:
# Kategorie-Code in Klartext umwandeln
df_codes = pd.read_excel("data/Mediznische Codezuordnung Berliner Feuerwehr_20260521.xlsx", sheet_name="AMPDS-Codes")
df_codes["kategorie_code"] = df_codes["Code"].str[:2]
kategorie_labels = df_codes[["kategorie_code", "Hauptbeschwerde_Text_Original"]].drop_duplicates()
print(df_codes.groupby("kategorie_code")["Hauptbeschwerde_Text_Original"].nunique().value_counts())
print(kategorie_labels.shape)

In [ ]:
# Bezirk x Dringlichkeitsstufe Aggregation (für die App)
district_spalten = [c for c in df_mission_data.columns if c.startswith("district_")]
criticality_spalten = [c for c in df_mission_data.columns if c.startswith("criticality_")]

zeilen = []
for d_col in district_spalten:
    bezirk = d_col.replace("district_", "")
    for c_col in criticality_spalten:
        crit = c_col.replace("criticality_", "")
        maske = df_mission_data[d_col] & df_mission_data[c_col]
        n = maske.sum()
        if n == 0:
            continue
        zeilen.append({
            "bezirk": bezirk,
            "criticality": crit,
            "n": n,
            "median_response_time": df_mission_data.loc[maske, "response_time"].median(),
            "hilfsfrist_quote": df_mission_data.loc[maske, "hilfsfrist"].mean(),
        })

df_bezirk_krit = pd.DataFrame(zeilen)
df_bezirk_krit = df_bezirk_krit[df_bezirk_krit["n"] >= 30]

print(df_bezirk_krit.shape)
df_bezirk_krit.head()

In [ ]:
kategorie_spalten = [c for c in df_mission_data.columns if c.startswith("kategorie_")]
district_spalten = [c for c in df_mission_data.columns if c.startswith("district_")]

zeilen = []
for d_col in district_spalten:
    bezirk = d_col.replace("district_", "")
    for k_col in kategorie_spalten:
        kategorie_raw = k_col.replace("kategorie_", "")
        maske = df_mission_data[d_col] & df_mission_data[k_col]
        zeilen.append({
            "bezirk": bezirk,
            "kategorie_raw": kategorie_raw,
            "n": maske.sum(),
            "median_response_time": df_mission_data.loc[maske, "response_time"].median(),
            "hilfsfrist_quote": df_mission_data.loc[maske, "hilfsfrist"].mean(),
        })

df_stats = pd.DataFrame(zeilen)

# n>=30 Schwelle (statistische Verlässlichkeit pro Bezirk x Kategorie)
df_stats = df_stats[df_stats["n"] >= 30]

# kategorie_raw ist "1.0", "10.0", ... oder "unbekannt" -> auf 2-stelligen Code formatieren
df_stats["kategorie_code"] = pd.to_numeric(df_stats["kategorie_raw"], errors="coerce")
df_stats = df_stats.dropna(subset=["kategorie_code"])  # wirft "unbekannt" raus
df_stats["kategorie_code"] = df_stats["kategorie_code"].astype(int).astype(str).str.zfill(2)

# Mit Labels mergen (inner join wirft automatisch die nicht gemappten Codes 35/37/52-90 raus)
df_stats = df_stats.merge(kategorie_labels, on="kategorie_code", how="inner")

print(df_stats.shape)
df_stats.head()

In [ ]:
import os
os.makedirs("app/data", exist_ok=True)
df_stats.to_csv("app/data/bezirk_kategorie_hilfsfrist.csv", index=False)

In [ ]:
df_bezirk_krit.to_csv("app/data/bezirk_kritikalitaet.csv", index=False)

In [ ]:
import joblib
joblib.dump(rf_reg, "app/data/rf_reg.pkl")
joblib.dump(feature_spalten, "app/data/feature_spalten.pkl")